# PCA in a Pipeline

This notebook shows how to use PCA as a preprocessing step inside a scikit-learn **pipeline**. We apply it to a face-recognition problem: each image has nearly 3,000 pixels, so we reduce them with PCA before training a classifier, and let a single pipeline handle both steps.

## Learning Objectives

At the end of this notebook, you should be able to:

- Build a scikit-learn pipeline that chains PCA with a logistic-regression classifier.
- Apply PCA to compress image data to a small set of components before classification.
- Tune the classifier with `GridSearchCV` and select the best hyperparameter by cross-validation.
- Evaluate the model on unseen faces with a classification report and a confusion matrix.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
sns.set()    # use seaborn plotting defaults
plt.rcParams["figure.figsize"] = (9, 6)    # set figure size for all plots 

## Example: Face Recognition

As an example of using PCA inside a pipeline, let's look at the facial recognition problem. We will use the [Labeled Faces in the Wild](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_lfw_people.html) dataset, which consists of several thousand collated photos of various public figures. A fetcher for the dataset is built into scikit-learn:

In [ ]:
from sklearn.datasets import fetch_lfw_people
faces = fetch_lfw_people(min_faces_per_person=60)
print(faces.target_names)
print(faces.images.shape)

Let's plot a few of these faces to see what we're working with:

In [ ]:
fig, ax = plt.subplots(3, 5)
for i, axi in enumerate(ax.flat):
    axi.imshow(faces.images[i], cmap='bone')
    axi.set(xticks=[], yticks=[],
            xlabel=faces.target_names[faces.target[i]])

Here we combine PCA with a **logistic regression** classifier, which you met earlier in the course. Logistic regression is a fast, well-understood linear model, and pairing it with PCA is a good illustration of dimensionality reduction as a preprocessing step: PCA turns the raw pixels into a compact set of features, and the classifier then learns from those.

One reason to reduce first is cost. Training directly on thousands of raw pixel features is slow and prone to overfitting, whereas 150 principal components keep the signal while being far cheaper to work with. This combination of dimensionality reduction and a classifier is what makes the pipeline below practical.


Each image contains [62×47] or nearly 3,000 pixels. We could proceed by simply using each pixel value as a feature, but it is often more effective to use a preprocessor to extract more meaningful features. Here we use a principal component analysis to extract 150 fundamental components to feed into our logistic-regression classifier. We can do this most straightforwardly by packaging the preprocessor and the classifier into a single pipeline:

The pipeline runs the two steps in order, and treats them as a single model to train and evaluate:

```mermaid
flowchart LR
    A[Face image: ~3000 pixels] --> B[PCA: 150 components]
    B --> C[Logistic Regression]
    C --> D[Predicted person]
```

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

pca = PCA(n_components=150, whiten=True, random_state=42)
clf = LogisticRegression(class_weight='balanced', max_iter=1000)
model = make_pipeline(pca, clf)

For the sake of testing our classifier output, we will split the data into a training and testing set:

In [ ]:
from sklearn.model_selection import train_test_split
Xtrain, Xtest, ytrain, ytest = train_test_split(faces.data, faces.target,
                                                random_state=42)

Finally, we can use a grid search with cross-validation to find a good value for the classifier's main hyperparameter.

For logistic regression that is ``C``, the inverse of the regularisation strength: a smaller ``C`` regularises more strongly (a simpler model), a larger ``C`` fits the training data more closely.

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {'logisticregression__C': [0.01, 0.1, 1, 10]}
grid = GridSearchCV(model, param_grid, verbose=5)

%time grid.fit(Xtrain, ytrain)
print(grid.best_params_)

The grid search prints the best value of `C` it found by cross-validation. Because it runs the full PCA-plus-classifier pipeline for every candidate, the PCA is refitted inside each fold on training data only, so no information leaks from the test set into the evaluation.

If the best value sits at an edge of the grid, widen the grid to be sure you have found the true optimum.

With this cross-validated model, we can now predict the labels for the test data, which the model has not yet seen:

In [ ]:
model = grid.best_estimator_
yfit = model.predict(Xtest)

Let's take a look at a few of the test images along with their predicted values:

In [ ]:
fig, ax = plt.subplots(4, 6)
for i, axi in enumerate(ax.flat):
    axi.imshow(Xtest[i].reshape(62, 47), cmap='bone')
    axi.set(xticks=[], yticks=[])
    axi.set_ylabel(faces.target_names[yfit[i]].split()[-1],
                   color='black' if yfit[i] == ytest[i] else 'red')
fig.suptitle('Predicted Names; Incorrect Labels in Red', size=14);

Most faces in this small sample are labelled correctly, with any mistakes shown in red above. We can get a better sense of the estimator's performance using the classification report, which lists recovery statistics label by label:

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(ytest, yfit,
                            target_names=faces.target_names))

The report shows precision, recall, and F1 for each person. Overall accuracy on the held-out faces is around 0.83, which is strong considering we compressed each image from nearly 3,000 pixels down to 150 components: most of the identifying information survived the reduction.

We might also display the confusion matrix between these classes:

In [ ]:
from sklearn.metrics import confusion_matrix
mat = confusion_matrix(ytest, yfit)
sns.heatmap(mat.T, square=True, annot=True, fmt='d', cbar=False,
            xticklabels=faces.target_names,
            yticklabels=faces.target_names)
plt.xlabel('true label')
plt.ylabel('predicted label');

This helps us get a sense of which labels are likely to be confused by the estimator.

For a real-world facial recognition task, in which the photos do not come pre-cropped into nice grids, the only difference in the facial classification scheme is the feature selection: you would need to use a more sophisticated algorithm to find the faces, and extract features that are independent of the pixelation.
For this kind of application, one good option is to make use of [OpenCV](http://opencv.org), which, among other things, includes pre-trained implementations of state-of-the-art feature extraction tools for images in general and faces in particular.

## Exercise: how many components?

We used 150 PCA components. Build the same pipeline with a different number of components (try 50 and 300 as well), refit it on the training data, and compare the test accuracy. Does using more components always help?

In [ ]:
from sklearn.metrics import accuracy_score

for n in [50, 150, 300]:
    pipe = make_pipeline(
        PCA(n_components=n, whiten=True, random_state=42),
        LogisticRegression(class_weight="balanced", max_iter=1000),
    )
    pipe.fit(Xtrain, ytrain)
    acc = accuracy_score(ytest, pipe.predict(Xtest))
    print(f"{n:3d} components: test accuracy = {acc:.3f}")

**Explanation**

Accuracy is about 0.73 with 50 components, rises to about 0.82 with 150, then falls back to about 0.79 with 300. More components are not always better: once the informative variance is captured, extra components mostly add noise for the classifier to overfit. Around 150 components is a good balance here.

## Summary

In this notebook you:

- Built a scikit-learn pipeline that chains PCA with a logistic-regression classifier.
- Applied PCA to compress each face from nearly 3,000 pixels to 150 components.
- Tuned the classifier with `GridSearchCV` and evaluated it on unseen faces.
- Reached about 0.83 accuracy, showing that dimensionality reduction kept most of the identifying signal.
- Compared the test accuracy across different numbers of PCA components.

## References & Further Reading

- [**scikit-learn: Pipeline**](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): Chaining preprocessing and a model into a single estimator.
- [**scikit-learn: PCA**](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html): The dimensionality-reduction step used here.
- [**scikit-learn: LogisticRegression**](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html): The classifier at the end of the pipeline.
- [**Labeled Faces in the Wild**](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_lfw_people.html): The face dataset used in this notebook.